In [1]:
# conda activate anndata

import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy import sparse
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

pd.set_option('display.max_columns', None)

In [2]:
plt.rcParams['figure.figsize'] = (9, 6)  # width, height in inches
plt.rcParams['figure.dpi'] = 200          # display DPI in notebook
plt.rcParams['savefig.dpi'] = 250   

In [3]:
psi = pd.read_csv("data/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_SJ_PSI.csv", index_col=0)

In [4]:
# Load single-cell PSI data
sdata = ad.read_h5ad("data/yao_2021_ACA_MOp_VISp_STAR_SJ_counts_PSI_annotated.h5ad")

# Load single-cell gene expression data
adata = ad.read_h5ad("data/yao_2021_ACA_MOp_VISp_STAR_model/yao_2021_ACA_MOp_VISp_STAR_gene_counts_scVI.h5ad")

adata.obs['subclass_label'] = adata.obs['subclass_label'].astype(str).str.replace("/", "_", regex=False).str.replace(" ", "_", regex=False)
sdata.obs['subclass_label'] = sdata.obs['subclass_label'].astype(str).str.replace("/", "_", regex=False).str.replace(" ", "_", regex=False)

# Work with full gene space
adata_raw = adata.raw.to_adata()
adata_raw.X = adata_raw.X.toarray()

In [5]:
adata.obs.value_counts('subclass_label')

subclass_label
L4_5_IT_CTX        3544
L6_CT_CTX          3425
Vip                2446
L6_IT_CTX          2353
Pvalb              1820
L2_3_IT_CTX        1779
Sst                1725
L5_IT_CTX          1579
Lamp5              1396
L6b_CTX            1169
L5_6_NP_CTX         882
L5_PT_CTX           498
Sncg                331
Astro               215
L2_3_IT_PPP         143
Sst_Chodl           100
Oligo                97
Endo                 69
SMC-Peri             49
Micro-PVM            46
VLMC                 42
Car3                 42
Meis2                38
L5_6_IT_TPE-ENT       8
CR                    4
L4_RSP-ACA            1
Name: count, dtype: int64

### Pairwise DE genes

In [6]:
def _safe(name):
    # simple filename sanitizer
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(name))

def _barplot_colors(ctypes, working_ctype):
    colors = ["tab:blue"] * len(ctypes)
    idx = pd.Index(ctypes).get_indexer([working_ctype])[0]
    colors[idx] = "tab:red"
    return colors

def _flatten_1d(x):
    if sparse.issparse(x):   # AnnData/sparse safe
        return x.toarray().ravel()
    return np.asarray(x, dtype=float).ravel()

def plot_barplot_by_cell_type(means, errs, ctypes, working_ctype, title, ylabel,
                              counts=None, show=True):
    means = np.asarray(means)
    errs = np.asarray(errs) * 2.0

    colors = _barplot_colors(ctypes, working_ctype)
    x = np.arange(len(ctypes))

    if counts is not None:
        # Two stacked panels: main barplot on top, cell counts on bottom
        fig, (ax, ax_n) = plt.subplots(
            2, 1,
            gridspec_kw={'height_ratios': [3, 1]},
        )
        # Top: main barplot
        ax.bar(x, means, yerr=errs, capsize=3, color=colors)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(ctypes, rotation=45, ha="right", fontsize=8)

        # Bottom: cell counts in gray
        ax_n.bar(x, counts, color="gray")
        ax_n.set_ylabel("# cells")
        ax_n.set_xticks(x)
        ax_n.set_xticklabels(ctypes, rotation=45, ha="right", fontsize=8)

        fig.tight_layout()
        fig.subplots_adjust(hspace=0.6)  # more space since both have rotated labels
    else:
        fig, ax = plt.subplots()
        ax.bar(x, means, yerr=errs, capsize=3, color=colors)
        ax.set_xticks(x)
        ax.set_xticklabels(ctypes, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        fig.tight_layout()

    if show:
        plt.show()
    return fig

def plot_ME_vs_PSI(mod_eig, exon_psi, title, show=True):
    fig, ax = plt.subplots()
    ax.scatter(mod_eig, exon_psi, s=18)
    ax.set_xlabel("Module eigengene")
    ax.set_ylabel("Exon PSI")
    ax.set_title(title)
    plt.tight_layout()
    if show:
        plt.show()
    return fig

def violin_with_points(values_by_ct, ctypes, focus=None,
                       base_fc='tab:blue', highlight_fc='tab:red',
                       ylabel="Value", title=None,
                       jitter=0.08, point_size=8, point_alpha=0.35,
                       max_points_per_ct=None, seed=0, violin_alpha=0.5,
                       show=False):
    rng = np.random.default_rng(seed)
    pos = np.arange(1, len(ctypes)+1)
    fig, ax = plt.subplots()
    vp = ax.violinplot(values_by_ct, positions=pos, showmeans=True, showextrema=True)
    # color violins (no outlines)
    for i, b in enumerate(vp['bodies']):
        col = highlight_fc if (focus is not None and ctypes[i] == focus) else base_fc
        b.set_facecolor(col)
        b.set_edgecolor('none')
        b.set_alpha(violin_alpha)
    for k in ('cmeans','cmins','cmaxes','cbars','cmedians'):
        if k in vp: vp[k].set_visible(False)
    # jitter points
    for i, v in enumerate(values_by_ct, start=1):
        v = np.asarray(v, float)
        v = v[np.isfinite(v)]
        if v.size == 0: continue
        if max_points_per_ct and v.size > max_points_per_ct:
            v = v[rng.choice(v.size, size=max_points_per_ct, replace=False)]
        x = i + (rng.random(v.size) - 0.5) * 2 * jitter
        col = highlight_fc if (focus is not None and ctypes[i-1] == focus) else base_fc
        ax.scatter(x, v, s=point_size, alpha=point_alpha, color=col, edgecolors='none', rasterized=True)
    ax.set_xticks(pos)
    ax.set_xticklabels(ctypes, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel(ylabel)
    if title: ax.set_title(title)
    fig.tight_layout()
    if show:
        plt.show()
    return fig

In [7]:
data_source = f"yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_strictFALSE_logFC3_nSignif21_enrichments"

psi = pd.read_csv("data/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_SJ_PSI.csv", index_col=0)
psi_corr_df = pd.read_csv(f"data/corrs/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_strictFALSE_logFC3_nSignif21_enrichments_PSI_vs_exon_corr.csv", index_col=0)
top_qval_mods_df = pd.read_csv("data/enrichments/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_yao_2021_ACA_MOp_VISp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_strictFALSE_logFC3_nSignif21_enrichments_abund_corr.csv")

In [8]:
# Remove genes correlated with abundance < .75

mask = ~(top_qval_mods_df['Old_cor'] < .75)
top_qval_mods_df = top_qval_mods_df.loc[mask]

In [9]:
top_qval_mods_df.index = top_qval_mods_df['Cell_type']
gene_exon_df = psi_corr_df['Gene']

In [10]:
np.unique(adata.obs['subclass_label'])

array(['Astro', 'CR', 'Car3', 'Endo', 'L2_3_IT_CTX', 'L2_3_IT_PPP',
       'L4_5_IT_CTX', 'L4_RSP-ACA', 'L5_6_IT_TPE-ENT', 'L5_6_NP_CTX',
       'L5_IT_CTX', 'L5_PT_CTX', 'L6_CT_CTX', 'L6_IT_CTX', 'L6b_CTX',
       'Lamp5', 'Meis2', 'Micro-PVM', 'Oligo', 'Pvalb', 'SMC-Peri',
       'Sncg', 'Sst', 'Sst_Chodl', 'VLMC', 'Vip'], dtype=object)

In [11]:
all_ctypes = [
    'CR', 'Car3', 'L2_3_IT_CTX', 'L2_3_IT_PPP',
    'L4_5_IT_CTX', 'L4_RSP-ACA', 'L5_6_IT_TPE-ENT', 'L5_6_NP_CTX',
    'L5_IT_CTX', 'L5_PT_CTX', 'L6_CT_CTX', 'L6_IT_CTX', 'L6b_CTX',
    'Pvalb', 'Sst', 'Sst_Chodl', 'Lamp5', 'Sncg', 'Vip', 'Meis2', 
    'Astro', 'Oligo', 'Endo', 'SMC-Peri', 'VLMC', 'Micro-PVM'
    ]

In [12]:
ctypes = psi_corr_df.columns[1:]

In [16]:
psi_corr_df.sort_values("Pvalb")

,Gene,Pvalb,L2_3_IT_CTX,L6b_CTX,Lamp5,L5_PT_CTX,Sncg,L5_6_NP_CTX,Sst_Chodl,Meis2,Oligo,Car3,VLMC,SMC-Peri,Astro,Endo,Micro-PVM
ENSMUSG00000033419_ProteinCoding_4,Snap91,-0.662547,0.171479,0.188508,-0.057885,0.031432,-0.098700,0.220246,0.000004,-0.019079,0.080407,0.132268,0.001879,0.007567,-0.070714,-0.001490,0.094445
ENSMUSG00000052889_NMD_1,Prkcb,-0.542513,0.286564,0.079922,-0.139660,0.066620,-0.177226,0.100035,-0.090252,-0.113625,0.131666,0.045551,-0.055740,0.020421,0.023753,0.038406,0.132435
ENSMUSG00000028478_ProteinCoding_4,Clta,-0.512870,0.275003,0.099801,-0.157646,0.075846,-0.351388,0.117726,-0.268127,-0.021118,0.013196,0.150508,0.023938,-0.071832,-0.030441,0.017601,0.167952
ENSMUSG00000022307_ProteinCoding_1,Oxr1,-0.503747,0.221930,0.130618,0.020669,0.169928,0.024701,0.173750,-0.273087,-0.060229,-0.025551,-0.003029,-0.079615,-0.093255,-0.037238,0.009895,0.079123
ENSMUSG00000026179_ProteinCoding_1,Pnkd,-0.485263,0.054468,-0.099781,-0.100414,-0.017224,-0.077535,0.140236,-0.073201,0.008930,0.304753,-0.009705,0.152868,0.333829,0.065473,0.210924,0.146168
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00000059534_other_1,Uqcr10,0.471633,-0.104932,-0.085447,-0.099654,0.052331,-0.013471,0.049020,0.153791,0.271897,-0.059304,-0.073074,-0.009636,-0.189719,0.066606,-0.147255,0.088085
ENSMUSG00000026825_ProteinCoding_1,Dnm1,0.478116,-0.397977,-0.000546,0.175436,-0.037141,0.298063,-0.054575,0.345336,0.138775,-0.091767,-0.067714,0.014986,0.026414,-0.007989,0.021249,-0.102796
ENSMUSG00000020436_ProteinCoding_1,Gabrg2,0.502908,0.000527,-0.364888,0.122098,-0.159537,0.036463,-0.058379,0.119155,-0.020265,0.111749,-0.162795,-0.009032,-0.022536,-0.019479,0.040339,0.097514
ENSMUSG00000026797_ProteinCoding_1,Stxbp1,0.637784,-0.308277,-0.167503,0.090584,-0.083813,0.184471,-0.274953,0.403791,-0.006138,-0.058241,-0.023961,-0.047225,0.008403,-0.060495,-0.035200,-0.117200


In [13]:
# # min_psi_corr = 0.25
# # ascending = True

# top_n = 15
# ascending = True

# for ctype in ctypes:
#     print(ctype)
#     outdir = f"figures/pseudobulk/{data_source}/{ctype}"
#     os.makedirs(outdir, exist_ok=True)
    
#     mask = top_qval_mods_df['Cell_type'] == ctype 
#     row = top_qval_mods_df[mask]
#     mod_df = pd.read_csv(row['Old_ME_path'].item())
#     mod_eig_df = mod_df.set_index("Sample")[row['Old_module'].item()]
#     mod_eig = pd.to_numeric(mod_eig_df, errors="coerce")

#     # Get candidate cell type-specific exons
    
#     # if ascending:
#     #     mask_is_max = (psi_corr_df.iloc[:, 1:].min(axis=1) == psi_corr_df[ctype]) # Subset to exons for which working cell type has the highest corr
#     #     mask_min_psi = (psi_corr_df[ctype] < -min_psi_corr)
#     # else:
#     #     mask_is_max = (psi_corr_df.iloc[:, 1:].max(axis=1) == psi_corr_df[ctype])
#     #     mask_min_psi = (psi_corr_df[ctype] > min_psi_corr)
        
#     # mask = mask_min_psi & mask_is_max

#     # if mask.sum() == 0:
#     #     continue
    
#     # psi_sorted = psi_corr_df[mask].sort_values(ctype, ascending=ascending)
#     psi_sorted = psi_corr_df.sort_values(ctype, ascending=ascending)[1:top_n]
#     top_exons = psi_sorted.index.values
    
#     # Get mean expression and variance of exon/gene in each cell type

#     cols = [f"{ct}_{stat}" for ct in ctypes for stat in ("mean", "SE")]
#     ctype_psi_df = pd.DataFrame(columns=cols, index=top_exons) 
#     ctype_expr_df = pd.DataFrame(columns=cols, index=top_exons)

#     for exon in top_exons:
#         exon_mask = sdata.var_names.isin([exon])
#         sdata_sub = sdata[:, exon_mask].copy()
#         gene_mask = adata_raw.var_names.isin([gene_exon_df.loc[exon]])
#         adata_sub = adata_raw[:, gene_mask].copy()
        
#         psi_vals_by_ct = []
#         expr_vals_by_ct = []
#         mean_psi_by_ct = [] 
#         mean_psi_se_by_ct = []
#         mean_expr_by_ct = []
#         mean_expr_se_by_ct = []

#         for ct in all_ctypes:
#             cell_mask = adata_sub.obs['subclass_label'] == ct
#             n = np.sum(cell_mask)

#             psi_per_cell = _flatten_1d(sdata_sub.X[cell_mask, :])
#             expr_per_cell = _flatten_1d(adata_sub.X[cell_mask, :])

#             psi_vals_by_ct.append(psi_per_cell)
#             expr_vals_by_ct.append(np.log1p(expr_per_cell))

#             mean_psi_by_ct.append(np.mean(psi_per_cell))
#             mean_psi_se_by_ct.append(np.sqrt(np.var(psi_per_cell) / n))

#             mean_expr = np.mean(adata_raw.X[cell_mask, :]) 
#             mean_expr_by_ct.append(np.mean(expr_per_cell))
#             mean_expr_se_by_ct.append(np.sqrt(np.var(expr_per_cell) / n) / mean_expr)

#         corr = round(psi_corr_df.loc[exon, ctype], 2)
#         gene = gene_exon_df.loc[exon]
#         exon_psi = psi.loc[exon]
#         exon_label = ''.join(str(exon).split("_")[1:])
#         # pdf_path = f"{outdir}/{_safe(ctype)}_{_safe(gene)}_{_safe(exon_label)}_minCor{min_psi_corr}_ascending{ascending}.pdf"
#         pdf_path = f"{outdir}/{_safe(ctype)}_{_safe(gene)}_{_safe(exon_label)}_top{top_n}_ascending{ascending}.pdf"

#         with PdfPages(pdf_path) as pdf:
#             fig = plot_barplot_by_cell_type(mean_psi_by_ct, mean_psi_se_by_ct, 
#                                             sc_ctypes, ctype,
#                                             title=f"PSI for {gene} {exon_label} exon",
#                                             ylabel="Mean PSI", show=False)
#             # pdf.savefig(fig); plt.close(fig)
#             plt.show()

#             fig = plot_barplot_by_cell_type(mean_expr_by_ct, mean_expr_se_by_ct, 
#                                             sc_ctypes, ctype,
#                                             title=f"Gene expression for {gene}",
#                                             ylabel="Mean expression (normalized)", 
#                                             show=False)
#             # pdf.savefig(fig); plt.close(fig)
#             plt.show()

#             fig = plot_ME_vs_PSI(mod_eig, exon_psi,
#                                  title=f"{ctype} ME vs. PSI for {gene} {exon_label} exon\nCorr: {corr}",
#                                  show=False)
#             # pdf.savefig(fig); plt.close(fig)
#             plt.show()


#             fig = violin_with_points(psi_vals_by_ct, sc_ctypes, focus=ctype,
#                                      ylabel="PSI",
#                                      title=f"PSI distribution for {gene} {exon_label} exon",
#                                      jitter=0.2, max_points_per_ct=3000, violin_alpha=0.2, 
#                                      show=False)
#             # pdf.savefig(fig); plt.close(fig)
#             plt.show()
            
#             fig = violin_with_points(expr_vals_by_ct, sc_ctypes, focus=ctype,
#                                      ylabel="Expression (log2)",
#                                      title=f"Count distribution for {gene}",
#                                      jitter=0.2, max_points_per_ct=3000, violin_alpha=0.2, 
#                                      show=False)
#             # pdf.savefig(fig); plt.close(fig)
#             plt.show()

#         print("Saved", exon)
#         print("")
#     break
    

In [17]:
top_n = 15
ascending = True 

for ctype in ctypes:
    print(ctype)
    outdir = f"figures/pseudobulk/{data_source}/{ctype}"
    os.makedirs(outdir, exist_ok=True)
    
    mask = top_qval_mods_df['Cell_type'] == ctype 
    row = top_qval_mods_df[mask]
    mod_df = pd.read_csv(row['Old_ME_path'].item())
    mod_eig_df = mod_df.set_index("Sample")[row['Old_module'].item()]
    mod_eig = pd.to_numeric(mod_eig_df, errors="coerce")

    psi_sorted = psi_corr_df.sort_values(ctype, ascending=ascending)[:top_n]
    top_exons = psi_sorted.index.values
    
    # Get mean expression and variance of exon/gene in each cell type
    cols = [f"{ct}_{stat}" for ct in ctypes for stat in ("mean", "SE")]
    ctype_psi_df = pd.DataFrame(columns=cols, index=top_exons) 
    ctype_expr_df = pd.DataFrame(columns=cols, index=top_exons)

    for exon in top_exons:
        exon_mask = sdata.var_names.isin([exon])
        sdata_sub = sdata[:, exon_mask].copy()
        gene_mask = adata_raw.var_names.isin([gene_exon_df.loc[exon]])
        adata_sub = adata_raw[:, gene_mask].copy()
        
        psi_vals_by_ct = []
        expr_vals_by_ct = []
        mean_psi_by_ct = [] 
        mean_psi_se_by_ct = []
        mean_expr_by_ct = []
        mean_expr_se_by_ct = []
        n_by_ct = []  # NEW: collect cell counts per type

        for ct in all_ctypes:
            cell_mask = adata_sub.obs['subclass_label'] == ct
            n = np.sum(cell_mask)
            n_by_ct.append(int(n))  # NEW

            psi_per_cell = _flatten_1d(sdata_sub.X[cell_mask, :])
            expr_per_cell = _flatten_1d(adata_sub.X[cell_mask, :])

            psi_vals_by_ct.append(psi_per_cell)
            expr_vals_by_ct.append(np.log1p(expr_per_cell))

            mean_psi_by_ct.append(np.mean(psi_per_cell))
            mean_psi_se_by_ct.append(np.sqrt(np.var(psi_per_cell) / n))

            mean_expr = np.mean(adata_raw.X[cell_mask, :]) 
            mean_expr_by_ct.append(np.mean(expr_per_cell))
            mean_expr_se_by_ct.append(np.sqrt(np.var(expr_per_cell) / n) / mean_expr)
    
        corr = round(psi_corr_df.loc[exon, ctype], 2)
        gene = gene_exon_df.loc[exon]
        exon_psi = psi.loc[exon]
        exon_label = ''.join(str(exon).split("_")[1:])
        pdf_path = f"{outdir}/{_safe(ctype)}_{_safe(gene)}_{_safe(exon_label)}_top{top_n}_ascending{ascending}.pdf"

        with PdfPages(pdf_path) as pdf:
            fig = plot_barplot_by_cell_type(mean_psi_by_ct, mean_psi_se_by_ct, 
                                            all_ctypes, ctype,
                                            title=f"PSI for {gene} {exon_label} exon",
                                            ylabel="Mean PSI",
                                            counts=n_by_ct,  # NEW
                                            show=False)
            pdf.savefig(fig); plt.close(fig)
            plt.show()

            fig = plot_barplot_by_cell_type(mean_expr_by_ct, mean_expr_se_by_ct, 
                                            all_ctypes, ctype,
                                            title=f"Gene expression for {gene}",
                                            ylabel="Mean expression (normalized)",
                                            counts=n_by_ct,  # NEW
                                            show=False)
            pdf.savefig(fig); plt.close(fig)
            plt.show()

            fig = plot_ME_vs_PSI(mod_eig, exon_psi,
                                 title=f"{ctype} module eigengene vs. PSI for {gene} {exon_label} exon\nCorr: {corr}",
                                 show=False)
            pdf.savefig(fig); plt.close(fig)
            plt.show()

            fig = violin_with_points(psi_vals_by_ct, all_ctypes, focus=ctype,
                                     ylabel="PSI",
                                     title=f"PSI distribution for {gene} {exon_label} exon",
                                     jitter=0.2, max_points_per_ct=3000, violin_alpha=0.2, 
                                     show=False)
            pdf.savefig(fig); plt.close(fig)
            plt.show()
            
            fig = violin_with_points(expr_vals_by_ct, all_ctypes, focus=ctype,
                                     ylabel="Expression (log2)",
                                     title=f"Count distribution for {gene}",
                                     jitter=0.2, max_points_per_ct=3000, violin_alpha=0.2, 
                                     show=False)
            pdf.savefig(fig); plt.close(fig)
            plt.show()

        print("Saved", gene, exon)
        print("")

Pvalb
Saved Snap91 ENSMUSG00000033419_ProteinCoding_4

Saved Prkcb ENSMUSG00000052889_NMD_1

Saved Clta ENSMUSG00000028478_ProteinCoding_4

Saved Oxr1 ENSMUSG00000022307_ProteinCoding_1

Saved Pnkd ENSMUSG00000026179_ProteinCoding_1

Saved Kcnc2 ENSMUSG00000035681_ProteinCoding_1

Saved Ndufa5 ENSMUSG00000023089_ProteinCoding_1

Saved Kcnc3 ENSMUSG00000062785_ProteinCoding_2

Saved Kcnip2 ENSMUSG00000025221_other_1

Saved Nptn ENSMUSG00000032336_ProteinCoding_1

Saved Kctd17 ENSMUSG00000033287_ProteinCoding_3

Saved Kctd17 ENSMUSG00000033287_NMD_1

Saved Ablim2 ENSMUSG00000029095_ProteinCoding_3

Saved Med28 ENSMUSG00000015804_ProteinCoding_2

Saved Cyrib ENSMUSG00000022378_ProteinCoding_1

L2_3_IT_CTX
Saved Ppp3ca ENSMUSG00000028161_ProteinCoding_1

Saved Gria3 ENSMUSG00000001986_NMD_1

Saved Dnm1 ENSMUSG00000026825_ProteinCoding_1

Saved Ndufb6 ENSMUSG00000071014_ProteinCoding_1

Saved Clstn1 ENSMUSG00000039953_ProteinCoding_1

Saved Ppp1r12a ENSMUSG00000019907_ProteinCoding_2

Saved

### Claude marker genes

In [50]:
# def _col_idx_for(df, stat):
#     return df.columns.str.contains(stat)

def _safe(name):
    # simple filename sanitizer
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(name))

def _barplot_colors(ctypes, working_ctype):
    colors = ["tab:blue"] * len(ctypes)
    idx = pd.Index(ctypes).get_indexer([working_ctype])[0]
    colors[idx] = "tab:red"
    return colors

def _flatten_1d(x):
    if sparse.issparse(x):   # AnnData/sparse safe
        return x.toarray().ravel()
    return np.asarray(x, dtype=float).ravel()

def plot_barplot_by_cell_type(means, errs, ctypes, title, ylabel, show=True):
    means = np.asarray(means)
    errs = np.asarray(errs) * 2.0 
    
    # colors = _barplot_colors(ctypes, working_ctype)
    x = np.arange(len(ctypes))
    fig, ax = plt.subplots()
    ax.bar(x, means, yerr=errs, capsize=3) # color=colors)
    ax.set_xticks(x); ax.set_xticklabels(ctypes, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel(ylabel); ax.set_title(title)
    fig.tight_layout()
    if show:
        plt.show()
    return fig

def plot_barplot_by_cell_type(means, errs, ctypes, title, ylabel,
                              counts=None, show=True):
    means = np.asarray(means)
    errs = np.asarray(errs) * 2.0

    # colors = _barplot_colors(ctypes, working_ctype)
    x = np.arange(len(ctypes))

    if counts is not None:
        # Two stacked panels: main barplot on top, cell counts on bottom
        fig, (ax, ax_n) = plt.subplots(
            2, 1,
            gridspec_kw={'height_ratios': [3, 1]},
        )
        # Top: main barplot
        ax.bar(x, means, yerr=errs, capsize=3) # , color=colors)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_xticks(x)
        ax.set_xticklabels(ctypes, rotation=45, ha="right", fontsize=8)

        # Bottom: cell counts in gray
        ax_n.bar(x, counts, color="gray")
        ax_n.set_ylabel("# cells")
        ax_n.set_xticks(x)
        ax_n.set_xticklabels(ctypes, rotation=45, ha="right", fontsize=8)

        fig.tight_layout()
        fig.subplots_adjust(hspace=0.6)  # more space since both have rotated labels
    else:
        fig, ax = plt.subplots()
        ax.bar(x, means, yerr=errs, capsize=3) # , color=colors)
        ax.set_xticks(x)
        ax.set_xticklabels(ctypes, rotation=45, ha="right", fontsize=8)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        fig.tight_layout()

    if show:
        plt.show()
    return fig


def plot_ME_vs_PSI(mod_eig, exon_psi, title, show=True):
    fig, ax = plt.subplots()
    ax.scatter(mod_eig, exon_psi, s=18)
    ax.set_xlabel("Module eigengene")
    ax.set_ylabel("Exon PSI")
    ax.set_title(title)
    plt.tight_layout()
    if show:
        plt.show()
    return fig

def violin_with_points(values_by_ct, ctypes, focus=None,
                       base_fc='tab:blue', highlight_fc='tab:red',
                       ylabel="Value", title=None,
                       jitter=0.08, point_size=8, point_alpha=0.35,
                       max_points_per_ct=None, seed=0, violin_alpha=0.5,
                       show=False):                         # <- default False when saving
    rng = np.random.default_rng(seed)
    pos = np.arange(1, len(ctypes)+1)

    fig, ax = plt.subplots()                                # <- create fig
    vp = ax.violinplot(values_by_ct, positions=pos, showmeans=True, showextrema=True)

    # color violins (no outlines)
    for i, b in enumerate(vp['bodies']):
        col = highlight_fc if (focus is not None and ctypes[i] == focus) else base_fc
        b.set_facecolor(col)
        b.set_edgecolor('none')
        b.set_alpha(violin_alpha)
    for k in ('cmeans','cmins','cmaxes','cbars','cmedians'):
        if k in vp: vp[k].set_visible(False)

    # jitter points
    for i, v in enumerate(values_by_ct, start=1):
        v = np.asarray(v, float)
        v = v[np.isfinite(v)]
        if v.size == 0: continue
        if max_points_per_ct and v.size > max_points_per_ct:
            v = v[rng.choice(v.size, size=max_points_per_ct, replace=False)]
        x = i + (rng.random(v.size) - 0.5) * 2 * jitter
        col = highlight_fc if (focus is not None and ctypes[i-1] == focus) else base_fc
        ax.scatter(x, v, s=point_size, alpha=point_alpha, color=col, edgecolors='none', rasterized=True)

    ax.set_xticks(pos); ax.set_xticklabels(ctypes, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel(ylabel); 
    if title: ax.set_title(title)
    fig.tight_layout()

    if show:
        plt.show()

    return fig  

In [51]:
data_source = f"yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_Claude_marker_genes_enrichments_ctype_abundance_PSI_vs_exon_corr"
psi_corr_df = pd.read_csv(f"data/corrs/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_Claude_marker_genes_enrichments_ctype_abundance_PSI_vs_exon_corr.csv", index_col=0)
mod_eig_df = pd.read_csv("data/yao_2021_ACA_MOp_VISp_STAR_SyntheticDataset1_10pcntCells_20SD_200samples_mergeParam0.95_subsetCutoff825.95_Modules_Claude_marker_genes_enrichments_ctype_abundance.csv", index_col=0)

In [52]:
gene_exon_df = psi_corr_df['Gene']

In [53]:
ctypes = mod_eig_df.columns

In [54]:
ctypes

Index(['All_GABAergic', 'All_Glutamatergic', 'All_Neuronal', 'Sst'], dtype='object')

In [57]:
top_n = 15
ascending = False 

for ctype in ctypes:
    print(ctype)
    outdir = f"figures/pseudobulk/{data_source}/{ctype}"
    os.makedirs(outdir, exist_ok=True)
    
    mod_eig = pd.to_numeric(mod_eig_df[ctype], errors="coerce") 
    mod_eig = mod_eig.reindex(psi.columns) 

    psi_sorted = psi_corr_df.sort_values(ctype, ascending=ascending)[:top_n]
    top_exons = psi_sorted.index.values
    
    # Get mean expression of cell type exon/gene in each cell type

    cols = [f"{ct}_{stat}" for ct in ctypes for stat in ("mean", "SE")]
    ctype_psi_df = pd.DataFrame(columns=cols, index=top_exons) 
    ctype_expr_df = pd.DataFrame(columns=cols, index=top_exons)

    for exon in top_exons:
        exon_mask = sdata.var_names.isin([exon])
        sdata_sub = sdata[:, exon_mask].copy()
        gene_mask = adata_raw.var_names.isin([gene_exon_df.loc[exon]])
        adata_sub = adata_raw[:, gene_mask].copy()
        
        psi_vals_by_ct = []
        expr_vals_by_ct = []
        mean_psi_by_ct = [] 
        mean_psi_se_by_ct = []
        mean_expr_by_ct = []
        mean_expr_se_by_ct = []

        for ct in all_ctypes:
            cell_mask = adata_sub.obs['subclass_label'] == ct
            n = sum(cell_mask)
            
            psi_per_cell = _flatten_1d(sdata_sub.X[cell_mask, :])
            expr_per_cell = _flatten_1d(adata_sub.X[cell_mask, :])

            psi_vals_by_ct.append(psi_per_cell)
            expr_vals_by_ct.append(np.log1p(expr_per_cell))

            mean_psi_by_ct.append(np.mean(psi_per_cell))
            mean_psi_se_by_ct.append(np.sqrt(np.var(psi_per_cell) / n))

            mean_expr = np.mean(adata_raw.X[cell_mask, :]) 
            mean_expr_by_ct.append(np.mean(expr_per_cell))
            mean_expr_se_by_ct.append(np.sqrt(np.var(expr_per_cell) / n) / mean_expr)

        corr = round(psi_corr_df.loc[exon, ctype], 2)
        gene = gene_exon_df.loc[exon]
        exon_psi = psi.loc[exon]
        exon_label = ''.join(str(exon).split("_")[1:])
        pdf_path = f"{outdir}/{_safe(ctype)}_{_safe(gene)}_{_safe(exon_label)}_ascending{ascending}.pdf"

        with PdfPages(pdf_path) as pdf:
            fig = plot_barplot_by_cell_type(mean_psi_by_ct, mean_psi_se_by_ct, 
                                            all_ctypes, 
                                            title=f"PSI for {gene} {exon_label} exon",
                                            ylabel="Mean PSI", show=False)
            pdf.savefig(fig); plt.close(fig)

            fig = plot_barplot_by_cell_type(mean_expr_by_ct, mean_expr_se_by_ct, 
                                            all_ctypes, 
                                            title=f"Gene expression for {gene}",
                                            ylabel="Mean expression (normalized)", 
                                            show=False)
            pdf.savefig(fig); plt.close(fig)

            fig = plot_ME_vs_PSI(mod_eig, exon_psi,
                                 title=f"{ctype} ME vs. PSI for {gene} {exon_label} exon\nCorr: {corr}",
                                 show=False)
            pdf.savefig(fig); plt.close(fig)

 
            fig = violin_with_points(psi_vals_by_ct, all_ctypes, focus=None,
                                     ylabel="PSI",
                                     title=f"PSI distribution for {gene} {exon_label} exon",
                                     jitter=0.2, max_points_per_ct=3000, violin_alpha=0.2, 
                                     show=False)
            pdf.savefig(fig); plt.close(fig)
            
            fig = violin_with_points(expr_vals_by_ct, all_ctypes, focus=None,
                                     ylabel="Expression (log2)",
                                     title=f"Count distribution for {gene}",
                                     jitter=0.2, max_points_per_ct=3000, violin_alpha=0.2, 
                                     show=False)
            pdf.savefig(fig); plt.close(fig)

        print("Saved", gene, exon)
        print("")
    

All_GABAergic
Saved Dnm1 ENSMUSG00000026825_ProteinCoding_1

Saved Stxbp1 ENSMUSG00000026797_ProteinCoding_1

Saved Rragb ENSMUSG00000041658_ProteinCoding_1

Saved Kif3a ENSMUSG00000018395_ProteinCoding_2

Saved Ndrg3 ENSMUSG00000027634_ProteinCoding_1

Saved Gtf2i ENSMUSG00000060261_ProteinCoding_6

Saved Sars1 ENSMUSG00000068739_ProteinCoding_1

Saved Cadm1 ENSMUSG00000032076_ProteinCoding_3

Saved Ncam1 ENSMUSG00000039542_ProteinCoding_1

Saved Dnm2 ENSMUSG00000033335_ProteinCoding_2

Saved Cyrib ENSMUSG00000022378_ProteinCoding_2

Saved Nrxn3 ENSMUSG00000066392_ProteinCoding_4

Saved Nrxn1 ENSMUSG00000024109_ProteinCoding_1

Saved Ewsr1 ENSMUSG00000009079_ProteinCoding_2

Saved Rpn2 ENSMUSG00000027642_ProteinCoding_2

All_Glutamatergic
Saved Nptn ENSMUSG00000032336_ProteinCoding_1

Saved Cltb ENSMUSG00000047547_ProteinCoding_1

Saved Cnih1 ENSMUSG00000015759_ProteinCoding_1

Saved Srsf9 ENSMUSG00000029538_NMD_1

Saved Itsn2 ENSMUSG00000020640_ProteinCoding_1

Saved Dync1i2 ENSMUSG0